# NISAR, Каховка v3: стійкий аналіз з урахуванням перешкод

Продовжує `nisar_kakhovka_timeseries.ipynb` і бере його кеш (`data/cache_ts`): 22 знімки, треки 143_A_025 і 108_D_065. Сюди вбудовано методи, які в базі конкурентів 2026 року працювали найкраще. Джерело кожного вказано в заголовку розділу.

| # | Ідея | Звідки | Що дає нам |
|---|---|---|---|
| 1 | Не змішувати версії обробки (CRID) | ismoil3 | прибрати штучний стрибок у кінці сезону |
| 2 | Багатодатний фільтр Quegan-Yu | Recycllux, obada-ab | менше спеклу без втрати меж полів |
| 3 | Надійність дати й категорія «не можна сказати» | monolyth365, rupokghosh, obada-ab | не рахувати перешкоди змінами |
| 4 | Адаптивний поріг води за постійною водою JRC | Recycllux, obada-ab | вода на кожну дату без навчання |
| 5 | Поріг max(3 дБ, 3σ) і зняття зсуву сцени | rupokghosh, yusof-shkn | поріг зміни від шуму самої пари |
| 6 | Плацебо на стабільному класі і conformal-поріг | BookieBun, rabbaniputrowi | перевірити, чи тест не бреше (наша H0) |
| 7 | Блоковий bootstrap і нуль-модель зсуву для двох треків | BookieBun, deniz2077 | чесні довірчі інтервали з урахуванням просторової залежності |
| 8 | Вердикти Supported / Not observable / … | AqeelDEV (changeproof) | відповідь замість карти |
| 9 | «Проблема хмар», виміряна на ділянці | Wawa29h, monolyth365 | чому радар, а не оптика |
| 10 | Кожне число з ID гранули (`facts.json`) | albaraahegaz, htjworld | відтворюваність для журі й демо |

Пороги в цьому ноутбуці взято з тих проєктів **до** перегляду наших результатів. Ми їх не підганяли.

## 0. Налаштування і завантаження кешу

In [ ]:
from pathlib import Path
from datetime import datetime
import os, re, json, warnings

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.ndimage import uniform_filter, median_filter, binary_erosion, binary_dilation, zoom
from scipy.stats import mannwhitneyu
from pyproj import Transformer
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from IPython.display import display

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 9})

DATA_DIR = Path('..') / 'data'
if not (DATA_DIR / 'h5').exists():
    DATA_DIR = next(p / 'data' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'h5').exists())
CACHE = DATA_DIR / 'cache_ts'
OUT = DATA_DIR / 'analysis_v3'; OUT.mkdir(exist_ok=True)
TRACKS = ['143_A_025', '108_D_065']; MAIN, ALT = TRACKS
BBOX, EPSG, RES = (33.4, 46.9, 35.2, 47.8), 32636, 80.0
rng = np.random.default_rng(0)

# Та сама сітка, що й у ноутбуці часового ряду
_t = Transformer.from_crs(4326, EPSG, always_xy=True)
_X, _Y = _t.transform([BBOX[0], BBOX[0], BBOX[2], BBOX[2]], [BBOX[1], BBOX[3], BBOX[1], BBOX[3]])
X = np.arange(np.floor((min(_X) - 40) / 80) * 80 + 40, max(_X), RES)
Y = np.arange(np.ceil((max(_Y) - 40) / 80) * 80 + 40, min(_Y), -RES)
H, W = len(Y), len(X)
TRANSFORM = from_origin(X[0] - RES / 2, Y[0] + RES / 2, RES, RES)
EXTENT = [(X[0] - 40) / 1e3, (X[-1] + 40) / 1e3, (Y[-1] - 40) / 1e3, (Y[0] + 40) / 1e3]
PIX_KM2 = RES**2 / 1e6

GRAN = {}
for p in (DATA_DIR / 'h5').glob('*.h5'):
    m = re.search(r'_(\d{3}_[AD]_\d{3})_\d{4}_\w{4}_\w_(\d{8}T\d{4})\d{2}_', p.name)
    if m: GRAN[(m.group(1), m.group(2))] = p.stem

S = {}
for t in TRACKS:
    files = sorted(CACHE.glob(f'{t}_*.npz'))
    assert files, 'Немає кешу: спершу запустіть nisar_kakhovka_timeseries.ipynb'
    d = {'time': [], 'hh': [], 'hv': [], 'nesz_hh': [], 'nesz_hv': [], 'version': [], 'granule': [], 'dswx': []}
    for f in files:
        key = f.stem.split('_')[-1]; tt = datetime.strptime(key, '%Y%m%dT%H%M')
        z = np.load(f); meta = json.loads(str(z['meta']))
        d['time'].append(tt); d['version'].append(meta['version']); d['granule'].append(GRAN.get((t, key), '?'))
        d['hh'].append(z['hhB'].astype('f4')); d['hv'].append(z['hvB'].astype('f4'))
        d['nesz_hh'].append(z['nesz_hhB'].astype('f4')); d['nesz_hv'].append(z['nesz_hvB'].astype('f4'))
        dz = CACHE / f'dswx_s1_{key}.npz'
        d['dswx'].append(np.load(dz)['mos'] if dz.exists() else np.full((H, W), 255, 'u1'))
    S[t] = {k: (np.stack(v) if k in ('hh', 'hv', 'nesz_hh', 'nesz_hv', 'dswx') else v) for k, v in d.items()}
    assert S[t]['hh'].shape[1:] == (H, W), 'кеш з іншою сіткою'
    print(f"{t}: {len(files)} дат, версії: {pd.Series(S[t]['version']).value_counts().to_dict()}")

def to_grid(src_path, resampling=Resampling.nearest, nodata=255):
    dst = np.full((H, W), nodata, 'u1')
    with rasterio.open(src_path) as src:
        reproject(rasterio.band(src, 1), dst, dst_transform=TRANSFORM, dst_crs=f'EPSG:{EPSG}',
                  src_nodata=src.nodata if src.nodata is not None else nodata, dst_nodata=nodata, resampling=resampling)
    return dst

WC = to_grid(DATA_DIR / 'aux' / 'worldcover_2021_aoi.tif', Resampling.mode, nodata=0)
JRC = to_grid(DATA_DIR / 'aux' / 'jrc_occurrence_aoi.tif', Resampling.average, nodata=255).astype('f4')
JRC[JRC > 100] = np.nan; J0 = np.nan_to_num(JRC, nan=0)
RES_BED = J0 >= 50
LAND_HIST = (J0 < 5) & (WC != 80)
DRY = (J0 == 0) & np.isin(WC, (30, 40))                      # суха суша для контрасту
ZONES = {'рілля': (WC == 40) & LAND_HIST, 'забудова': (WC == 50) & LAND_HIST,
         'дно: дерева': RES_BED & (WC == 10), 'дно: трави/чагарники/болото': RES_BED & np.isin(WC, (20, 30, 90)), 'дно: усе': RES_BED}

## 1. Версії обробки (ідея ismoil3)

Останні дати мають іншу версію продукту (1.0.20, VALIDATED), ніж решта (1.0.11, PROVISIONAL). ismoil3 знайшов між версіями зсув близько −0.6 дБ HH. Перевіряємо на **забудові**, яка за сезон фізично не змінюється. Висновок про «кінець сезону» робимо у двох варіантах: з усіма датами і лише з датами основної версії.

In [ ]:
rows = []
for t in TRACKS:
    for i, tt in enumerate(S[t]['time']):
        rows.append({'трек': t, 'дата': tt, 'версія': S[t]['version'][i],
                     'HH забудова': np.nanmedian(S[t]['hh'][i][ZONES['забудова']]), 'HV забудова': np.nanmedian(S[t]['hv'][i][ZONES['забудова']]),
                     'HH рілля': np.nanmedian(S[t]['hh'][i][ZONES['рілля']])})
VER = pd.DataFrame(rows)
vrows = []
for t in TRACKS:
    q = VER[VER['трек'] == t]; vs = sorted(q['версія'].unique())
    if len(vs) < 2: continue
    a, b = q[q['версія'] == vs[0]], q[q['версія'] == vs[-1]]
    for c in ('HH забудова', 'HV забудова', 'HH рілля'):
        vrows.append({'трек': t, 'ряд': c, f'медіана {vs[0]}': a[c].median(), f'медіана {vs[-1]}': b[c].median(),
                      'зсув нова − стара, дБ': b[c].median() - a[c].median(), 'n дат': f'{len(a)} / {len(b)}',
                      'p (Манн-Вітні)': mannwhitneyu(a[c], b[c]).pvalue})
VERSION_TABLE = pd.DataFrame(vrows); display(VERSION_TABLE.round(3))
MAIN_VERSION = {t: pd.Series(S[t]['version']).mode()[0] for t in TRACKS}
print('Основна версія:', MAIN_VERSION, '— вікна сезону «лише основна версія» будуються з неї.')

## 2. Багатодатний фільтр спеклу Quegan-Yu (ідея Recycllux, obada-ab)

$J_k = \langle I_k\rangle \cdot \frac{1}{N}\sum_i I_i / \langle I_i\rangle$: локальне середнє 7×7 кожної дати множиться на середнє за всі дати нормоване відбиття. Спекл гаситься за рахунок часу, а не простору, тому межі полів лишаються різкими.

In [ ]:
def quegan(stack_db, win=7):
    I = 10 ** (stack_db / 10); valid = np.isfinite(I); I0 = np.where(valid, I, 0).astype('f4')
    E = np.stack([uniform_filter(I0[k], win) / np.maximum(uniform_filter(valid[k].astype('f4'), win), 1e-6) for k in range(len(I0))])
    ratio = np.where(valid, I0 / np.maximum(E, 1e-12), 0).sum(0) / np.maximum(valid.sum(0), 1)
    return np.where(valid, 10 * np.log10(np.maximum(E * ratio[None], 1e-12)), np.nan).astype('f4')

for t in TRACKS:
    S[t]['hhF'], S[t]['hvF'] = quegan(S[t]['hh']), quegan(S[t]['hv'])

def local_std(a, w=5):
    a0 = np.nan_to_num(a, nan=np.nanmedian(a)); m = uniform_filter(a0, w)
    return np.sqrt(np.maximum(uniform_filter(a0**2, w) - m**2, 0))
crop_core = binary_erosion(ZONES['рілля'], iterations=3)
for t in TRACKS:
    i = len(S[t]['time']) // 2
    print(f"{t} ({S[t]['time'][i]:%d.%m}): локальний розкид HH 5×5 на ріллі {np.nanmedian(local_std(S[t]['hh'][i])[crop_core]):.2f} → "
          f"{np.nanmedian(local_std(S[t]['hhF'][i])[crop_core]):.2f} дБ після фільтра")
win = (slice(560, 900), slice(700, 1150))
fig, axs = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
i = len(S[MAIN]['time']) // 2
for ax, k, ttl in ((axs[0], 'hh', 'до фільтра'), (axs[1], 'hhF', 'після Quegan-Yu')):
    ax.imshow(S[MAIN][k][i][win], cmap='gray', vmin=-20, vmax=-2); ax.set_title(f'HH {MAIN} {S[MAIN]["time"][i]:%d.%m}, {ttl}'); ax.set_axis_off()
fig.savefig(OUT / 'quegan_before_after.png', dpi=120); plt.show()

## 3. Надійність кожної дати і карта «що взагалі видно» (ідеї obada-ab, monolyth365, rupokghosh)

* **Еталон води:** ядро води, яка є водою за DSWx-S1 на ≥ 90 % дат обох треків, з відступом від берега. Це один сезонний шар, а не мітка конкретної дати.
* **Контраст (obada-ab):** суха суша мінус еталон води в HH. Якщо менше 3 дБ, воду від суші на цю дату не відрізнити.
* **Зсув сцени (monolyth365):** медіанна зміна сухої суші проти попередньої дати. Якщо більша за половину порога змін (1.5 дБ), змінилась уся сцена, тобто перешкоди або калібрування.
* **Чи видно піксель (за ідеєю rupokghosh «cannot tell»):** справжня суша на L-band має HH − HV ≥ 3 дБ, а адитивна перешкода дає HH ≈ HV. Тому піксель **видно**, якщо медіана HH − HV у вікні 5×5 ≥ 3 дБ і HH хоча б на 3 дБ вищий за NESZ. Інакше категорія — «не можна сказати».

> **Виправлення після першого прогону (15.11).** (1) Спочатку еталоном була JRC ≥ 95 %. Але JRC описує 1984–2021 роки, коли водосховище було повним, тож «еталон» виявився дном, яке тепер поросло лісом (1569 км²). (2) Рівень шуму рахувався як 5-й перцентиль HH у блоці 2 км, що записувало темні поля в «шум» (видно лише 23 % AOI в HH і 3 % у HV). Обидва місця замінено, як описано вище. Пороги 3 дБ не змінювались.

In [ ]:
_wat = np.zeros((H, W), 'f4'); _val = np.zeros((H, W), 'f4')
for t in TRACKS:
    _wat += np.isin(S[t]['dswx'], (1, 3)).sum(0); _val += (S[t]['dswx'] < 250).sum(0)
ALWAYS_WATER = (_val >= 10) & (_wat / np.maximum(_val, 1) >= 0.9)
for it in (3, 2, 1, 0):
    CORE = binary_erosion(ALWAYS_WATER, iterations=it) if it else ALWAYS_WATER
    if CORE.sum() * PIX_KM2 >= 10: break
print(f'Ядро води (DSWx-S1 ≥ 90 % дат): {CORE.sum() * PIX_KM2:.0f} км² (відступ {it} пікс.); для порівняння JRC ≥ 95 %: {(J0 >= 95).sum() * PIX_KM2:.0f} км²')

BLK = 25
def block_stat(a, fn):
    h, w = (H // BLK) * BLK, (W // BLK) * BLK
    v = a[:h, :w].reshape(h // BLK, BLK, w // BLK, BLK).transpose(0, 2, 1, 3).reshape(h // BLK, w // BLK, -1)
    out = fn(v); out[np.isfinite(v).sum(-1) < BLK * BLK * 0.2] = np.nan
    return out

def upsample(c):
    c = pd.DataFrame(c).interpolate(axis=1, limit_direction='both').interpolate(axis=0, limit_direction='both').to_numpy()
    up = zoom(np.nan_to_num(c, nan=np.nanmedian(c)), BLK, order=1)
    full = np.empty((H, W), 'f4'); full[:up.shape[0], :up.shape[1]] = up
    full[up.shape[0]:, :] = full[up.shape[0] - 1:up.shape[0], :]; full[:, up.shape[1]:] = full[:, up.shape[1] - 1:up.shape[1]]
    return full

REL = []
for t in TRACKS:
    T = S[t]; n = len(T['time']); T['obs_hh'] = np.zeros((n, H, W), bool); T['obs_hv'] = np.zeros((n, H, W), bool)
    for i in range(n):
        ratio = median_filter(np.nan_to_num(T['hhF'][i] - T['hvF'][i], nan=0), 5)
        obs = np.isfinite(T['hhF'][i]) & np.isfinite(T['hvF'][i]) & (ratio >= 3) & (T['hhF'][i] > T['nesz_hh'][i] + 3)
        T['obs_hh'][i] = obs; T['obs_hv'][i] = obs
        ref_hh, ref_hv = np.nanmedian(T['hhF'][i][CORE]), np.nanmedian(T['hvF'][i][CORE])
        dry_hh, dry_hv = np.nanmedian(T['hhF'][i][DRY]), np.nanmedian(T['hvF'][i][DRY])
        shift = np.nanmedian((T['hhF'][i] - T['hhF'][i - 1])[DRY]) if i else 0.0
        REL.append({'трек': t, 'дата': T['time'][i], 'версія': T['version'][i], 'вода HH': ref_hh, 'вода HV': ref_hv,
                    'суша HH': dry_hh, 'суша HV': dry_hv, 'контраст HH, дБ': dry_hh - ref_hh, 'контраст HV, дБ': dry_hv - ref_hv,
                    'зсув сцени HH, дБ': shift, 'видно HH, % AOI': 100 * T['obs_hh'][i].mean(), 'видно HV, % AOI': 100 * T['obs_hv'][i].mean(),
                    'видно HH рілля, %': 100 * T['obs_hh'][i][ZONES['рілля']].mean()})
REL = pd.DataFrame(REL)
REL['надійна'] = (REL['контраст HH, дБ'] >= 3) & (REL['зсув сцени HH, дБ'].abs() <= 1.5)
for t in TRACKS:
    S[t]['good'] = REL[REL['трек'] == t]['надійна'].to_numpy()
display(REL.round(1))
print('Надійних дат:', {t: f"{S[t]['good'].sum()} з {len(S[t]['good'])}" for t in TRACKS})

fig, axs = plt.subplots(1, 2, figsize=(15, 4), constrained_layout=True)
for t, mk in zip(TRACKS, 'os'):
    q = REL[REL['трек'] == t]
    axs[0].plot(q['дата'], q['контраст HH, дБ'], mk + '-', label=t)
    axs[1].plot(q['дата'], q['видно HH, % AOI'], mk + '-', label=t)
axs[0].axhline(3, c='k', ls='--', lw=.8); axs[0].set(title='Контраст суша − постійна вода (HH): нижче 3 дБ воду не відрізнити', ylabel='дБ'); axs[0].legend()
axs[1].set(title='Частка AOI, яку видно (HH − HV ≥ 3 дБ і HH ≥ NESZ + 3 дБ)', ylabel='%'); axs[1].legend(fontsize=8)
fig.savefig(OUT / 'reliability_by_date.png', dpi=130); plt.show()

## 4. Вода на кожну дату адаптивним порогом (ідеї Recycllux і obada-ab) і звірка з DSWx-S1

* **R (Recycllux):** вода, якщо HH < еталон + 1.5 дБ і HV < еталон + 2 дБ, у межах домену JRC (історична вода + 2 пікселі). Еталон — ядро постійної води на цю дату.
* **M (obada-ab):** поріг посередині між еталоном води і сухою сушею в HH і HV; дату пропускаємо, якщо контраст < 3 дБ.

Жоден метод не бачить DSWx. Базова лінія — класифікатор із попереднього ноутбука, навчений на DSWx з інших дат.

In [ ]:
DOMAIN = binary_dilation(J0 > 0, iterations=2)
def scores(pred, truth, m):
    p, w = pred[m], truth[m]; tp, fp, fn = (p & w).sum(), (p & ~w).sum(), (~p & w).sum()
    return tp / max(tp + fp + fn, 1), tp / max(tp + fp, 1), tp / max(tp + fn, 1)

rows = []
for t in TRACKS:
    T = S[t]; T['water'] = np.zeros((len(T['time']), H, W), bool)
    for i in range(len(T['time'])):
        r = REL[(REL['трек'] == t)].iloc[i]
        hh, hv = T['hhF'][i], T['hvF'][i]
        wR = (hh < r['вода HH'] + 1.5) & (hv < r['вода HV'] + 2) & DOMAIN
        thr_hh, thr_hv = (r['вода HH'] + r['суша HH']) / 2, (r['вода HV'] + r['суша HV']) / 2
        wM = (hh < thr_hh) & (hv < thr_hv) & DOMAIN
        T['water'][i] = wM
        truth = np.isin(T['dswx'][i], (1, 3)); m = DOMAIN & (T['dswx'][i] < 250) & np.isfinite(hh)
        for name, w in (('R: еталон + допуск', wR), ('M: середина вода–суша', wM)):
            iou, pr, rc = scores(w, truth, m) if m.sum() else (np.nan,) * 3
            rows.append({'трек': t, 'дата': T['time'][i], 'надійна': T['good'][i], 'метод': name, 'IoU': iou, 'точність': pr, 'повнота': rc,
                         'вода NISAR на дні, км²': (w & RES_BED).sum() * PIX_KM2, 'вода DSWx на дні, км²': (truth & RES_BED & (T['dswx'][i] < 250)).sum() * PIX_KM2})
WATER = pd.DataFrame(rows)
base = DATA_DIR / 'analysis_timeseries' / 'water_by_date.csv'
if base.exists():
    b = pd.read_csv(base, parse_dates=['час']).rename(columns={'час': 'дата'})
    b = b.merge(REL[['трек', 'дата', 'надійна']], on=['трек', 'дата'])
    WATER = pd.concat([WATER, b.assign(метод='класифікатор (база, v2)')[['трек', 'дата', 'надійна', 'метод', 'IoU']]], ignore_index=True)
summary = WATER.groupby(['метод', 'надійна'])['IoU'].agg(['median', 'max', 'count']).round(3)
print('IoU з DSWx-S1: надійні дати проти ненадійних'); display(summary)
display(WATER[WATER['метод'].str.startswith('M')].round(2))
WATER.to_csv(OUT / 'water_adaptive.csv', index=False, encoding='utf-8-sig')

fig, axs = plt.subplots(1, 2, figsize=(15, 4.3), constrained_layout=True)
for t, mk in zip(TRACKS, 'os'):
    q = WATER[(WATER['трек'] == t) & WATER['метод'].str.startswith('M')]
    axs[0].plot(q['дата'], q['вода DSWx на дні, км²'], mk + '-', c='tab:blue', label=f'DSWx-S1 ({t[:5]})')
    g = q[q['надійна']]; axs[0].plot(q['дата'], q['вода NISAR на дні, км²'], mk + ':', c='tab:orange', alpha=.4)
    axs[0].plot(g['дата'], g['вода NISAR на дні, км²'], mk, c='tab:orange', ms=8, label=f'NISAR M, надійні дати ({t[:5]})')
    axs[1].scatter(REL[REL['трек'] == t]['контраст HH, дБ'], q['IoU'], marker=mk, label=t)
axs[0].set(title='Вода на дні: DSWx-S1 і NISAR (великі маркери — надійні дати)', ylabel='км²'); axs[0].legend(fontsize=8)
axs[1].axvline(3, c='k', ls='--', lw=.8); axs[1].set(xlabel='контраст суша − вода HH, дБ', ylabel='IoU з DSWx-S1', title='Чи передбачає контраст якість води?'); axs[1].legend()
fig.savefig(OUT / 'water_adaptive.png', dpi=130); plt.show()

## 5. Зміни між сусідніми датами: поріг max(3 дБ, 3σ) і «не можна сказати» (ідеї rupokghosh, yusof-shkn)

Для кожної пари: Δ = нова − стара (після Quegan-Yu), мінус медіанний зсув сухої суші. σ = 1.4826·MAD(Δ) на видимих пікселях. Поріг = max(3 дБ, 3σ). Класи: зросло / впало / без змін / **не можна сказати** (піксель не видно хоча б на одній із дат).

In [ ]:
PAIRS = []
CLASS_MAPS = {}
for t in TRACKS:
    T = S[t]; maps = []
    for i in range(len(T['time']) - 1):
        d = T['hhF'][i + 1] - T['hhF'][i]; d = d - np.nanmedian(d[DRY])
        obs = T['obs_hh'][i] & T['obs_hh'][i + 1]
        z = d[obs & np.isfinite(d)]; sigma = 1.4826 * np.median(np.abs(z - np.median(z))) if z.size else np.nan
        thr = max(3.0, 3 * sigma)
        cls = np.select([~obs, d >= thr, d <= -thr], [0, 2, 1], 3)
        maps.append(cls)
        valid = np.isfinite(T['hhF'][i]) & np.isfinite(T['hhF'][i + 1])
        PAIRS.append({'трек': t, 'пара': f"{T['time'][i]:%d.%m}–{T['time'][i + 1]:%d.%m}", 'надійна пара': bool(T['good'][i] and T['good'][i + 1]),
                      'σ, дБ': sigma, 'поріг, дБ': thr, 'не можна сказати, %': 100 * (cls == 0)[valid].mean(),
                      'впало, %': 100 * (cls == 1)[valid].mean(), 'зросло, %': 100 * (cls == 2)[valid].mean(),
                      'рілля змінилась, % видимої': 100 * np.isin(cls, (1, 2))[ZONES['рілля'] & obs].mean(),
                      'забудова змінилась, % видимої': 100 * np.isin(cls, (1, 2))[ZONES['забудова'] & obs].mean()})
    CLASS_MAPS[t] = np.stack(maps)
PAIRS = pd.DataFrame(PAIRS); display(PAIRS.round(2))

cm4 = ListedColormap(['#bdbdbd', '#2b6cff', '#ff3b2b', '#ffffff'])
for t in TRACKS:
    M = CLASS_MAPS[t]; n = len(M); nc = 5; nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.4 * nc, 2.6 * nr), constrained_layout=True)
    for k, ax in enumerate(axs.ravel()):
        ax.set_axis_off()
        if k < n:
            ax.imshow(M[k][::2, ::2], cmap=cm4, vmin=0, vmax=3, interpolation='nearest', extent=EXTENT)
            r = PAIRS[(PAIRS['трек'] == t)].iloc[k]; ax.set_title(f"{r['пара']}{' ✓' if r['надійна пара'] else ''}  сіре {r['не можна сказати, %']:.0f}%")
    fig.suptitle(f'{t}: сіре — не можна сказати, синє — впало, червоне — зросло (поріг max(3 дБ, 3σ)); ✓ — обидві дати надійні')
    fig.savefig(OUT / f'pair_changes_{t}.png', dpi=110); plt.show()

## 6. Плацебо і conformal-поріг: чи тест не бреше? (ідеї BookieBun, rabbaniputrowi; наша H0)

Забудова за 12 днів не змінюється, тож будь-яка «зміна» там — хибна тривога. Для кожної пари рахуємо поріг |Δ| як 95-й перцентиль забудови **на всіх інших парах** (leave-one-pair-out conformal) і дивимось, яку частку хибних тривог він дає на відкладеній парі. Якщо метод стабільний, частка має лежати в межах [2.5 %, 10 %] (BookieBun: [α/2, 2α]). Той самий поріг на ріллі показує, наскільки рілля «живіша» за плацебо.

In [ ]:
ALPHA = 0.05
CONF = []
for t in TRACKS:
    T = S[t]; dl = []
    for i in range(len(T['time']) - 1):
        d = T['hhF'][i + 1] - T['hhF'][i]; d = d - np.nanmedian(d[DRY]); obs = T['obs_hh'][i] & T['obs_hh'][i + 1]
        dl.append((np.abs(d[ZONES['забудова'] & obs]), np.abs(d[ZONES['рілля'] & obs]), bool(T['good'][i] and T['good'][i + 1])))
    for subset in ('усі пари', 'лише надійні'):
        idx = [k for k in range(len(dl)) if subset == 'усі пари' or dl[k][2]]
        for k in idx:
            others = [dl[j][0] for j in idx if j != k and dl[j][0].size]
            if not others or not dl[k][0].size: continue
            q = np.quantile(np.concatenate(others), 1 - ALPHA)
            far = np.mean(dl[k][0] > q); crop = np.mean(dl[k][1] > q) if dl[k][1].size else np.nan
            CONF.append({'трек': t, 'набір': subset, 'пара': PAIRS[PAIRS['трек'] == t].iloc[k]['пара'], 'поріг |Δ|, дБ': q,
                         'хибні тривоги на забудові': far, 'у межах [α/2, 2α]': ALPHA / 2 <= far <= 2 * ALPHA, 'рілля понад поріг': crop,
                         'рілля / плацебо': crop / max(far, 1e-3)})
CONF = pd.DataFrame(CONF)
display(CONF.groupby(['трек', 'набір']).agg(пар=('пара', 'size'), у_межах=('у межах [α/2, 2α]', 'sum'),
                                             FAR_мін=('хибні тривоги на забудові', 'min'), FAR_макс=('хибні тривоги на забудові', 'max'),
                                             рілля_до_плацебо_медіана=('рілля / плацебо', 'median')).round(3))
CONF.to_csv(OUT / 'conformal_placebo.csv', index=False, encoding='utf-8-sig')

## 7. Стійкі сезонні зміни: два треки, блоковий bootstrap і нуль-модель зсуву (ідеї BookieBun, deniz2077)

**Вікна вибираємо за якістю, а не за результатом:** «початок» — 3 дати червня–липня з найбільшою видимою часткою AOI (розділ 3), «кінець» — 3 такі дати вересня–жовтня. Надійних за строгим правилом дат лише по одній на трек, тож вимагати їх у вікнах неможливо. Варіанти: з усіма версіями і лише з основною.

**Чутливість до видимості (ідея rupokghosh: діапазон замість одного числа):** піксель береться, якщо його видно хоча б на 1 (м'яко) або на 2 (строго) датах кожного вікна в обох треках.

> **Виправлення після першого прогону.** Спершу вікнами були перші й останні три дати. У 108_D 21.06–27.07 видно лише 1–6 % AOI, тож перетин видимих пікселів звівся до ~36 км².

Перевірки кореляції між треками:
* **Блоковий bootstrap** (блоки 5 км) дає 95 % інтервал r з урахуванням того, що сусідні пікселі залежні.
* **Нуль-модель зсуву:** карту другого треку циклічно зсуваємо на ≥ 10 км 200 разів. Просторова структура зберігається, а збіг місць руйнується. p = частка зсувів, де r не менший за справжній.

In [ ]:
def windows(t, pure):
    T = S[t]; vis = REL[REL['трек'] == t]['видно HH, % AOI'].to_numpy()
    cand = [i for i in range(len(T['time'])) if not pure or T['version'][i] == MAIN_VERSION[t]]
    early = [i for i in cand if T['time'][i].month in (6, 7)]; late = [i for i in cand if T['time'][i].month in (9, 10)]
    top = lambda idx: sorted(sorted(idx, key=lambda i: -vis[i])[:3])
    return top(early), top(late)

def persist(t, pol, pure, min_obs=2):
    T = S[t]; a, b = windows(t, pure)
    A = np.where(T[f'obs_{pol}'][a], T[f'{pol}F'][a], np.nan); B = np.where(T[f'obs_{pol}'][b], T[f'{pol}F'][b], np.nan)
    ok = (np.isfinite(A).sum(0) >= min_obs) & (np.isfinite(B).sum(0) >= min_obs)
    return np.where(ok, np.nanmedian(B, 0) - np.nanmedian(A, 0), np.nan), a, b

BLOCK = 62                                   # 62 × 80 м ≈ 5 км
BID = (np.arange(H)[:, None] // BLOCK) * (W // BLOCK + 1) + (np.arange(W)[None, :] // BLOCK)

def block_boot(values_list, mask, stat, n=500, sample=60000):
    idx = np.flatnonzero(mask)
    if idx.size == 0: return np.nan, (np.nan, np.nan)
    if idx.size > sample: idx = rng.choice(idx, sample, replace=False)
    vals = [v.ravel()[idx] for v in values_list]; bid = BID.ravel()[idx]
    ub, inv = np.unique(bid, return_inverse=True); groups = [np.flatnonzero(inv == g) for g in range(len(ub))]
    est = stat(*vals); boots = []
    for _ in range(n):
        pick = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        boots.append(stat(*[v[pick] for v in vals]))
    return est, tuple(np.nanpercentile(boots, [2.5, 97.5]))

corr = lambda a, b: np.corrcoef(a, b)[0, 1]
def fmt_win(t, a, b):
    f = lambda idx: ', '.join(f"{S[t]['time'][i]:%d.%m}" for i in idx)
    return f'{f(a)} → {f(b)}'
bg = {t: upsample(np.nanmedian(np.stack([block_stat(x, lambda v: np.nanmedian(v, axis=-1)) for x in np.where(LAND_HIST, S[t]['hv'], np.nan)]), 0)) for t in TRACKS}
RFI_ZONE = np.zeros((H, W), bool)
for t in TRACKS:
    RFI_ZONE |= (bg[t] > np.nanmedian(bg[t][LAND_HIST]) + 3) & (np.isfinite(S[t]['hh']).mean(0) > 0.5)   # без екстраполяції за край кадру
print(f'Зона постійної плями перешкод: {RFI_ZONE.mean():.0%} AOI')

XT, PERS = [], {}
for pure in (False, True):
  for min_obs in (1, 2):
    for pol in ('hh', 'hv'):
        pa, a1, b1 = persist(MAIN, pol, pure, min_obs); pb, a2, b2 = persist(ALT, pol, pure, min_obs); PERS[(pure, min_obs, pol)] = (pa, pb)
        for zname, zm in (('рілля', ZONES['рілля']), ('рілля поза плямою', ZONES['рілля'] & ~RFI_ZONE), ('дно: усе', RES_BED)):
            m = zm & np.isfinite(pa) & np.isfinite(pb)
            if m.sum() < 500:
                XT.append({'версії': 'лише основна' if pure else 'усі', 'видимість': f'≥{min_obs}', 'поляр.': pol.upper(), 'зона': zname, 'км²': m.sum() * PIX_KM2}); continue
            r, ci = block_boot([pa, pb], m, corr)
            null = []
            for _ in range(200):
                dy, dx = rng.integers(125, H - 125), rng.integers(125, W - 125)
                pbs = np.roll(np.roll(pb, dy, 0), dx, 1); mm = zm & np.isfinite(pa) & np.isfinite(pbs)
                if mm.sum() > 1000: null.append(np.corrcoef(pa[mm], pbs[mm])[0, 1])
            XT.append({'версії': 'лише основна' if pure else 'усі', 'видимість': f'≥{min_obs}', 'поляр.': pol.upper(), 'зона': zname, 'км²': m.sum() * PIX_KM2,
                       'r': r, 'r 95% CI (блоки 5 км)': f'[{ci[0]:.2f}, {ci[1]:.2f}]', 'r нуль-моделі p95': np.nanpercentile(null, 95),
                       'p (зсув)': (np.sum(np.array(null) >= r) + 1) / (len(null) + 1),
                       'вікна 143_A': fmt_win(MAIN, a1, b1), 'вікна 108_D': fmt_win(ALT, a2, b2)})
XT = pd.DataFrame(XT); display(XT.round(3))
XT.to_csv(OUT / 'cross_track_robust.csv', index=False, encoding='utf-8-sig')

pa, pb = PERS[(True, 1, 'hh')]
agree = np.select([(pa > 2) & (pb > 2), (pa < -2) & (pb < -2), np.isfinite(pa) & np.isfinite(pb)], [2, 1, 3], 0)
fig, ax = plt.subplots(figsize=(9.5, 6.8), constrained_layout=True)
ax.imshow(np.nanmedian(S[MAIN]['hhF'], 0), cmap='gray', vmin=-18, vmax=-2, extent=EXTENT)
ax.imshow(np.ma.masked_where(~np.isin(agree, (1, 2)), agree), cmap=ListedColormap(['#2b6cff', '#ff3b2b']), vmin=1, vmax=2, extent=EXTENT, interpolation='nearest')
ax.contour(RFI_ZONE[::-1], [0.5], colors='yellow', linewidths=.8, extent=EXTENT)
ax.set_title('Обидва треки (основна версія, видно ≥ 1 дати у вікні): HH кінця сезону нижче (синє) / вище (червоне) > 2 дБ; жовте — пляма перешкод')
fig.savefig(OUT / 'cross_track_agreement_robust.png', dpi=130); plt.show()
print(f"Узгоджено: зниження {(agree == 1).sum() * PIX_KM2:,.0f} км², зростання {(agree == 2).sum() * PIX_KM2:,.0f} км² "
      f"з {np.isin(agree, (1, 2, 3)).sum() * PIX_KM2:,.0f} км², де видно обидва треки")

## 8. Вердикти по зонах (ідея AqeelDEV / changeproof)

Питання для кожної зони: **«чи змінилось відбиття між початком і кінцем сезону?»** Вікна лише з основною версією; медіана Δ з блоковим 95 % інтервалом окремо для кожного треку.
* **Not observable (не видно):** видимих пікселів у зоні менше 25 % в **обох** треках (поріг 50 % у changeproof стосувався покриття буфера знімком; у нас частка видимих пікселів, тож беремо м'якше і пишемо це прямо).
* **Supported (зміну підтверджено):** обидва інтервали не містять 0, знак однаковий і |медіана| ≥ 0.5 дБ.
* **Contradicted (суперечать):** обидва інтервали не містять 0, але знаки різні.
* **Not supported (зміни не виявлено):** обидва інтервали містять 0.
* **Not provable (недоказово):** решта випадків; окремо позначаємо, коли зону видно лише одним треком, і що цей трек бачить.

In [ ]:
VERD = []
for min_obs in (1, 2):
  for pol in ('hh', 'hv'):
    pa, pb = PERS[(True, min_obs, pol)]
    for zname, zm in ZONES.items():
        r = {'видимість': f'≥{min_obs}', 'зона': zname, 'поляр.': pol.upper()}
        cis = []
        for t, p in ((MAIN, pa), (ALT, pb)):
            share = np.isfinite(p)[zm].mean(); est, ci = block_boot([p], zm & np.isfinite(p), lambda v: np.median(v), n=300)
            r[f'видно {t[:5]}, %'] = 100 * share; r[f'Δ {t[:5]}, дБ'] = est; r[f'CI {t[:5]}'] = f'[{ci[0]:+.2f}, {ci[1]:+.2f}]'; cis.append((share, est, ci))
        (s1, e1, c1), (s2, e2, c2) = cis
        ex1, ex2 = not (c1[0] <= 0 <= c1[1]), not (c2[0] <= 0 <= c2[1])
        if max(s1, s2) < 0.25: v = 'Not observable — не видно'
        elif min(s1, s2) < 0.25:
            (s, e, c), name = ((s1, e1, c1), MAIN) if s1 >= s2 else ((s2, e2, c2), ALT)
            seen = not (c[0] <= 0 <= c[1]) and abs(e) >= 0.5
            v = f'Not provable — видно лише {name}' + (f' (Δ {e:+.1f} дБ)' if seen else ', зміни не видно')
        elif ex1 and ex2 and np.sign(e1) == np.sign(e2) and min(abs(e1), abs(e2)) >= 0.5: v = 'Supported — зміну підтверджено'
        elif ex1 and ex2 and np.sign(e1) != np.sign(e2): v = 'Contradicted — треки суперечать'
        elif not ex1 and not ex2: v = 'Not supported — зміни не виявлено'
        else: v = 'Not provable — недоказово'
        r['вердикт'] = v; VERD.append(r)
VERD = pd.DataFrame(VERD)
asof = max(S[t]['time'][windows(t, True)[1][-1]] for t in TRACKS)
print(f'Вердикти станом на {asof:%Y-%m-%d %H:%M} UTC (останній знімок у вікнах кінця сезону)')
display(VERD.round(2))
print('Зведення вердиктів (м\'яка / строга видимість):')
display(VERD.pivot_table(index=['зона', 'поляр.'], columns='видимість', values='вердикт', aggfunc='first'))
VERD.to_csv(OUT / 'verdicts.csv', index=False, encoding='utf-8-sig')

## 9. «Проблема хмар» на самій ділянці (ідеї Wawa29h, monolyth365)

Для кожного дня з оптичним знімком (OPERA DSWx-HLS: Landsat 8/9, Sentinel-2) рахуємо частку AOI без хмар. Поруч — частка AOI, яку NISAR **справді бачить** за розділом 3. Виходить чесне порівняння: оптика сліпне від хмар, NISAR тут — від перешкод.

In [ ]:
HLS = {}
for p in sorted((DATA_DIR / 'opera_dswx' / 'hls').glob('*_B01_WTR.tif')):
    day = re.search(r'_T\w{5}_(\d{8})T', p.name).group(1)
    cf = CACHE / f'hls_{p.stem[:60]}.npy'
    a = np.load(cf) if cf.exists() else to_grid(p, Resampling.nearest)
    if not cf.exists(): np.save(cf, a)
    covered, clear = a != 255, a < 250
    if day in HLS: HLS[day] = (HLS[day][0] | covered, HLS[day][1] | clear)
    else: HLS[day] = (covered, clear)
CLOUD = pd.DataFrame([{'день': pd.Timestamp(datetime.strptime(d, '%Y%m%d')), 'покрито AOI, %': 100 * c.mean(), 'ясно AOI, %': 100 * k.mean()} for d, (c, k) in sorted(HLS.items())])
good_days = CLOUD[CLOUD['ясно AOI, %'] >= 50]['день']
gaps = good_days.diff().dt.days.dropna()
print(f"Днів з оптичним знімком: {len(CLOUD)}; з них ≥ 50 % AOI без хмар: {len(good_days)}; найдовша перерва між такими днями: {gaps.max():.0f} діб")
fig, ax = plt.subplots(figsize=(14, 3.8), constrained_layout=True)
ax.bar(CLOUD['день'], CLOUD['ясно AOI, %'], width=1, color='tab:green', alpha=.6, label='оптика: AOI без хмар, %')
for t, mk in zip(TRACKS, 'os'):
    q = REL[REL['трек'] == t]; ax.plot(q['дата'], q['видно HH, % AOI'], mk, c='tab:purple', ms=7, label=f'NISAR {t}: AOI видно, %')
ax.axhline(50, c='k', ls='--', lw=.7); ax.set_ylabel('%'); ax.legend(fontsize=8); ax.set_title('Що бачить оптика і що бачить NISAR над Каховкою, червень–листопад 2026')
fig.savefig(OUT / 'clouds_vs_nisar.png', dpi=130); plt.show()

## 10. Підсумок і `facts.json` з ID гранул (ідеї albaraahegaz, htjworld)

Кожне число нижче записано в `facts.json` разом з ID гранул, з яких воно пораховане, щоб демо і слайди брали числа звідти, а не з рук.

In [ ]:
def granules(t, idx): return [S[t]['granule'][i] for i in idx]
xt = XT.set_index(['версії', 'видимість', 'поляр.', 'зона'])
FACTS = {
    'version_offset': VERSION_TABLE.to_dict('records'),
    'reliable_dates': {t: [f'{S[t]["time"][i]:%Y-%m-%d}' for i in range(len(S[t]['time'])) if S[t]['good'][i]] for t in TRACKS},
    'water_iou_median': {f'{m} | надійна={g}': float(v) for (m, g), v in WATER.groupby(['метод', 'надійна'])['IoU'].median().items()},
    'conformal_within_band': {f'{t} | {s}': int(v) for (t, s), v in CONF.groupby(['трек', 'набір'])['у межах [α/2, 2α]'].sum().items()},
    'cross_track': XT.drop(columns=['вікна 143_A', 'вікна 108_D'], errors='ignore').to_dict('records'),
    'verdicts': VERD[['видимість', 'зона', 'поляр.', 'вердикт']].to_dict('records'),
    'optical_days': int(len(CLOUD)), 'optical_clear_days_50pct': int(len(good_days)), 'optical_longest_gap_days': float(gaps.max()) if len(gaps) else None,
    'granules': {t: {'start': granules(t, windows(t, True)[0]), 'end': granules(t, windows(t, True)[1]), 'all': S[t]['granule']} for t in TRACKS},
    'as_of_utc': f'{asof:%Y-%m-%dT%H:%M}',
}
(OUT / 'facts.json').write_text(json.dumps(FACTS, ensure_ascii=False, indent=1, default=str), encoding='utf-8')

L = []
for _, r in VERSION_TABLE[VERSION_TABLE['ряд'] == 'HH забудова'].iterrows():
    L.append(f"1. Версії ({r['трек']}): забудова HH зсунута на {r['зсув нова − стара, дБ']:+.2f} дБ у новій версії (n {r['n дат']}, p = {r['p (Манн-Вітні)']:.2f}).")
L.append('2. Надійних дат (контраст ≥ 3 дБ і зсув сцени ≤ 1.5 дБ): ' + ', '.join(f"{t}: {S[t]['good'].sum()} з {len(S[t]['good'])}" for t in TRACKS) + '.')
L.append(f"3. AOI, яку NISAR бачить (HH − HV ≥ 3 дБ і HH ≥ NESZ + 3): медіана {REL['видно HH, % AOI'].median():.0f} % (від {REL['видно HH, % AOI'].min():.0f} до {REL['видно HH, % AOI'].max():.0f} %).")
ws = WATER.groupby(['метод', 'надійна'])['IoU'].median()
L.append('4. Вода проти DSWx-S1, медіана IoU (надійні / ненадійні дати): ' + '; '.join(f"{m}: {ws.get((m, True), np.nan):.2f} / {ws.get((m, False), np.nan):.2f}" for m in WATER['метод'].unique()) + '.')
cs = CONF.groupby(['трек', 'набір']).agg(n=('пара', 'size'), ok=('у межах [α/2, 2α]', 'sum'), ratio=('рілля / плацебо', 'median'))
L.append('5. Conformal-поріг на плацебо (забудова), пар у межах [2.5 %, 10 %] хибних тривог: ' + '; '.join(f"{t} {s}: {int(r.ok)}/{int(r.n)}, рілля/плацебо ×{r.ratio:.1f}" for (t, s), r in cs.iterrows()) + '.')
for mo in ('≥1', '≥2'):
    for z in ('рілля', 'рілля поза плямою', 'дно: усе'):
        r = xt.loc[('лише основна', mo, 'HH', z)]
        if pd.notna(r.get('r', np.nan)):
            L.append(f"6. Два треки, {z} (HH, основна версія, видимість {mo}): {r['км²']:,.0f} км², r = {r['r']:.2f} {r['r 95% CI (блоки 5 км)']}, нуль-модель p95 = {r['r нуль-моделі p95']:.2f}, p = {r['p (зсув)']:.3f}.")
        else:
            L.append(f"6. Два треки, {z} (видимість {mo}): лише {r['км²']:,.0f} км² видно в обох треках, r не рахуємо.")
for mo in ('≥1', '≥2'):
    q = VERD[VERD['видимість'] == mo]
    L.append(f'7. Вердикти (видимість {mo}): ' + '; '.join(f"{r['зона']} {r['поляр.']}: {r['вердикт'].split(' — ')[0]}" for _, r in q.iterrows()) + '.')
L.append(f"8. Оптика: {len(CLOUD)} днів зі знімком, ≥ 50 % AOI без хмар лише {len(good_days)}; найдовша перерва {gaps.max():.0f} діб.")
print('\n'.join(L)); (OUT / 'summary.txt').write_text('\n'.join(L), encoding='utf-8')
print('\nЗбережено:', OUT)

## 11. Висновки v3

**Що змінилось від впровадження ідей конкурентів**
1. **Версії обробки (ismoil3).** У 143_A забудова HH у версії 1.0.20 на +1.06 дБ вища, ніж у 1.0.11 (лише 2 дати, p = 0.08). Тому висновки про сезон будуємо лише з основної версії 1.0.11. Без цього кінець сезону мав би штучний зсув.
2. **Quegan-Yu.** Локальний розкид HH на ріллі зменшився з 1.10 до 0.83 дБ (143_A) і з 0.83 до 0.48 дБ (108_D).
3. **Надійність дат.** За правилами obada-ab і monolyth365 надійна лише одна дата на трек (23.08 і 20.08): лише тоді постійна вода хоча б на 3 дБ темніша за сушу. Видимість («не можна сказати» за HH − HV) коливається від 1 до 65 % AOI. У 108_D у липні видно лише 1–6 %.
4. **Вода (Recycllux, obada-ab).** Адаптивний поріг без навчання дає IoU ≈ 0.25 з DSWx-S1 проти 0.11–0.14 у класифікатора з v2. Краще, але далеко від 0.6–0.9, які отримують на чистих сценах.
5. **Плацебо + conformal (BookieBun).** Частка хибних тривог на забудові з порогом, відкаліброваним на інших парах, лежить у межах [2.5 %, 10 %] лише в 1 з 10 пар кожного треку (від 0 до 50–98 %). Тест між парами дат **не стабільний**: це наша H0, і її провал прямо записуємо. Водночас рілля перевищує поріг у 14–18 разів частіше, ніж забудова.
6. **Два треки з чесною статистикою (BookieBun, deniz2077).** Стійка зміна HH ріллі «літо → осінь» корелює між треками: r = 0.43 [0.36, 0.51] (≥ 1 видима дата, 444 км²) і r = 0.69 [0.61, 0.77] (≥ 2 дати, 58 км²). Нуль-модель зсуву дає максимум 0.21–0.28, p = 0.005. Поза плямою перешкод так само. **Це головний підтверджений результат:** сезонна зміна полів реальна. Спостерігається переважно падіння HH (118 км² узгодженого зниження проти 7 км² зростання), що відповідає збиранню врожаю й оранці.
7. **Вердикти (changeproof).** Жодна зона не отримує «Supported»: 108_D бачить < 25 % кожної зони. За 143_A рілля HH −0.9 дБ, забудова HH без зміни (плацебо пройдено). HV падає **скрізь**, включно із забудовою (−1.8 дБ), тож сезонне падіння HV — дрейф сцени, а не поверхня.
8. **Хмари (Wawa29h).** З 58 днів з оптичним знімком лише 15 мають ≥ 50 % AOI без хмар, найдовша перерва 27 діб. NISAR бачить 1–65 % AOI на дату. Отже, **жоден сенсор сам не покриває сезон**, і це чесний сюжет для презентації.

**Обмеження, які лишаються**
* На парних картах змін ще трапляються великі розмиті плями (наприклад, 11.08–23.08 і 23.08–04.09 на південному сході). Їх фільтр видимості не ловить, тому попарні карти не використовуємо для висновків — лише сезонні медіани з перевіркою двома треками.
* DSWx-S1 і WorldCover — моделі, а не наземна правда. Ядро води взято з DSWx за весь сезон, тож перевірка води по DSWx частково залежна (позначено в розділі 3).
* Виправлення після першого прогону (еталон води, видимість, вікна) записано в розділах 3 і 7. Пороги 3 дБ / 1.5 дБ / 25 % не підганялися під результат.